# 10 — Delta Lake & Unity Catalog Operations

**What you will learn:**
- Create managed and external Delta tables
- CRUD: INSERT, UPDATE, DELETE on Delta tables
- MERGE (upsert) — the most important Delta operation
- Time Travel: read previous versions and restore
- Schema Evolution: add/change columns
- OPTIMIZE and Z-ORDER: compaction and data skipping
- VACUUM: delete old file versions
- Change Data Feed (CDF): capture row-level changes
- Unity Catalog: three-part naming, GRANT/REVOKE

## Setup — Base DataFrames

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from pyspark.sql.functions import col, current_timestamp, lit
import pyspark.sql.functions as F

# Employee source data
emp_data = [
    (1, "Alice",   "Engineering", 95000.0,  "2022-01-15", "Active"),
    (2, "Bob",     "Marketing",   72000.0,  "2021-06-01", "Active"),
    (3, "Charlie", "Engineering", 105000.0, "2020-03-20", "Active"),
    (4, "Diana",   "HR",          68000.0,  "2023-09-10", "Active"),
    (5, "Eve",     "Marketing",   78000.0,  "2022-11-05", "Active"),
]

df_emp = spark.createDataFrame(
    emp_data, ["emp_id", "name", "department", "salary", "hire_date", "status"]
)

## 1. Create a Managed Delta Table

In [0]:
# Write as managed Delta table (storage managed by Databricks)
df_emp.write.format("delta").mode("overwrite").saveAsTable("default.employees")

# Inspect with SQL
spark.sql("DESCRIBE TABLE default.employees").show(truncate=False)

In [0]:
spark.sql("SELECT * FROM default.employees ORDER BY emp_id").show()

## 3. INSERT — Add New Rows

In [0]:
# INSERT using SQL
spark.sql("""
    INSERT INTO default.employees VALUES
        (6, 'Frank',  'Engineering', 88000.0, '2021-04-12', 'Active'),
        (7, 'Grace',  'HR',          71000.0, '2022-07-30', 'Active')
""")

spark.sql("SELECT * FROM default.employees ORDER BY emp_id").show()

In [0]:
# INSERT using DataFrame API (append mode)
new_emp = spark.createDataFrame(
    [(8, "Hank", "Engineering", 92000.0, "2023-01-01", "Active")],
    ["emp_id", "name", "department", "salary", "hire_date", "status"]
)
new_emp.write.format("delta").mode("append").saveAsTable("default.employees")

spark.sql("SELECT * FROM default.employees ORDER BY emp_id").show()

## 4. UPDATE — Modify Existing Rows

In [0]:
# UPDATE using SQL (DML — requires Delta)
spark.sql("""
    UPDATE default.employees
    SET salary = salary * 1.10,
        status = 'Active'
    WHERE department = 'Engineering'
""")

spark.sql("SELECT emp_id, name, department, salary FROM default.employees ORDER BY emp_id").show()

In [0]:
# UPDATE using DeltaTable API (Python)
from delta.tables import DeltaTable

delta_tbl = DeltaTable.forName(spark, "default.employees")

delta_tbl.update(
    condition = col("department") == "HR",
    set       = {"salary": col("salary") * lit(1.05)}
)

spark.sql("SELECT emp_id, name, department, salary FROM default.employees ORDER BY emp_id").show()

## 5. DELETE — Remove Rows

In [0]:
# DELETE using SQL
spark.sql("""
    DELETE FROM default.employees
    WHERE status = 'Inactive'
""")

# DELETE using DeltaTable API
delta_tbl.delete(condition = col("emp_id") == 8)

spark.sql("SELECT * FROM default.employees ORDER BY emp_id").show()

## 6. MERGE (Upsert) — The Core Delta Operation

MERGE matches source records against target and:
- **MATCHED** → UPDATE or DELETE
- **NOT MATCHED BY TARGET** → INSERT
- **NOT MATCHED BY SOURCE** → DELETE (soft-deletes)

In [0]:
# Incoming changes: some updates + some new employees
changes_data = [
    (2,  "Bob",     "Marketing",   75000.0, "2021-06-01", "Active"),   # salary updated
    (4,  "Diana",   "Finance",     72000.0, "2023-09-10", "Active"),   # dept changed
    (5,  "Eve",     "Marketing",   78000.0, "2022-11-05", "Resigned"), # status change
    (9,  "Ivy",     "Marketing",   76000.0, "2024-01-10", "Active"),   # new employee
    (10, "Jake",    "Engineering", 98000.0, "2024-02-01", "Active"),   # new employee
]
df_changes = spark.createDataFrame(
    changes_data, ["emp_id", "name", "department", "salary", "hire_date", "status"]
)
df_changes.show()

In [0]:
# Perform MERGE
delta_tbl = DeltaTable.forName(spark, "default.employees")

(
    delta_tbl.alias("target")
    .merge(
        df_changes.alias("source"),
        "target.emp_id = source.emp_id"      # match key
    )
    .whenMatchedUpdate(set={
        "name":       "source.name",
        "department": "source.department",
        "salary":     "source.salary",
        "status":     "source.status",
    })
    .whenNotMatchedInsert(values={
        "emp_id":     "source.emp_id",
        "name":       "source.name",
        "department": "source.department",
        "salary":     "source.salary",
        "hire_date":  "source.hire_date",
        "status":     "source.status",
    })
    .execute()
)

spark.sql("SELECT * FROM default.employees ORDER BY emp_id").show()

## 7. Time Travel — Read Previous Versions

In [0]:
# Show full history of the table
spark.sql("DESCRIBE HISTORY default.employees").show(truncate=False)

In [0]:
# Read a specific version by number
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table("default.employees")
print("Version 0 (original):")
df_v0.show()

In [0]:
# Read as of a specific timestamp
# df_ts = spark.read.format("delta").option("timestampAsOf", "2024-01-01").table("default.employees")

# Time travel using SQL syntax
spark.sql("SELECT * FROM default.employees VERSION AS OF 0").show()

In [0]:
# Restore a table to a previous version
spark.sql("RESTORE TABLE default.employees TO VERSION AS OF 0")
spark.sql("SELECT * FROM default.employees ORDER BY emp_id").show()

## 8. Schema Evolution

In [0]:
# By default, Delta rejects writes with extra/different columns
# Enable mergeSchema to add new columns automatically

new_data_with_extra_col = spark.createDataFrame(
    [(11, "Kate", "Finance", 82000.0, "2024-03-01", "Active", "MBA")],
    ["emp_id", "name", "department", "salary", "hire_date", "status", "education"]
)

(
    new_data_with_extra_col.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")    # allows adding new column
    .saveAsTable("default.employees")
)

spark.sql("SELECT * FROM default.employees ORDER BY emp_id").show()

In [0]:
# For overwrite with full schema replacement, use overwriteSchema
# (only use this when you intentionally want to change the schema)
# df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(...)

## 9. OPTIMIZE and Z-ORDER

In [0]:
# OPTIMIZE — compact many small files into larger files (improves read performance)
spark.sql("OPTIMIZE default.employees")

In [0]:
# Z-ORDER — sort data within files by frequently filtered columns
# Files with matching values are co-located, enabling data skipping
spark.sql("OPTIMIZE default.employees ZORDER BY (department, emp_id)")

# After ZORDER, queries like:
# SELECT * FROM employees WHERE department = 'Engineering'
# will skip files that don't contain Engineering rows

## 10. VACUUM — Clean Up Old Files

In [0]:
# VACUUM removes data files no longer referenced by Delta transaction log
# Default retention = 7 days (168 hours)
# Files newer than retention period are NOT deleted (protects time travel)

spark.sql("VACUUM default.employees")

# To reduce retention for dev (NOT recommended in production — breaks time travel)
# spark.sql("VACUUM default.employees RETAIN 0 HOURS")  # requires spark.databricks.delta.retentionDurationCheck.enabled=false

## 11. Change Data Feed (CDF)

CDF records every row-level INSERT, UPDATE, and DELETE as change events.
Useful for: incremental pipelines, CDC to downstream systems.

In [0]:
# Enable CDF on the table
spark.sql("""
    ALTER TABLE default.employees
    SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

enable_version = (
    spark.sql("DESCRIBE HISTORY default.employees")
    .filter("operation = 'SET TBLPROPERTIES'")
    .selectExpr("max(version) as version")
    .first()["version"]
)

# Make a change
spark.sql("UPDATE default.employees SET salary = salary + 1000 WHERE department = 'Marketing'")

# Read the change feed — shows what changed and _change_type
cdf_df = (
    spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", enable_version)
    .table("default.employees")
)
cdf_df.select("emp_id", "name", "salary", "_change_type", "_commit_version").show()

## 12. Unity Catalog — Three-Part Naming & Governance

In [0]:
# Unity Catalog uses three-part naming: catalog.schema.table
# Create catalog and schema (requires Metastore Admin or USAGE + CREATE privilege)

spark.sql("CREATE CATALOG IF NOT EXISTS dev_catalog COMMENT 'Dev environment'")
spark.sql("USE CATALOG dev_catalog")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Raw layer'")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Cleansed layer'")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold   COMMENT 'Aggregated layer'")

In [0]:
# Create a Delta table in Unity Catalog
df_emp.write.format("delta").mode("overwrite").saveAsTable("dev_catalog.bronze.employees")

# Query with full three-part name
spark.sql("SELECT * FROM dev_catalog.bronze.employees ORDER BY emp_id").show()

## 13. Grants & Privileges in Unity Catalog

In [0]:
# Grant catalog access to a user
# spark.sql("GRANT USAGE ON CATALOG dev_catalog TO `user@company.com`")

# Grant schema access
# spark.sql("GRANT USAGE ON SCHEMA dev_catalog.bronze TO `data-engineers`")

# Grant SELECT on table
# spark.sql("GRANT SELECT ON TABLE dev_catalog.bronze.employees TO `data-analysts`")

# Grant MODIFY (insert/update/delete)
# spark.sql("GRANT MODIFY ON TABLE dev_catalog.bronze.employees TO `data-engineers`")

# Grant CREATE TABLE on schema
# spark.sql("GRANT CREATE TABLE ON SCHEMA dev_catalog.silver TO `data-engineers`")

# View existing grants
spark.sql("SHOW GRANTS ON TABLE dev_catalog.bronze.employees").show(truncate=False)

## 14. Delta Table Properties Reference

In [0]:
# View table properties
spark.sql("SHOW TBLPROPERTIES dev_catalog.bronze.employees").show(truncate=False)

# Set useful table properties
spark.sql("""
    ALTER TABLE dev_catalog.bronze.employees
    SET TBLPROPERTIES (
        'delta.enableChangeDataFeed'    = 'true',
        'delta.autoOptimize.optimizeWrite' = 'true',   -- auto-compact on write
        'delta.autoOptimize.autoCompact'   = 'true',   -- background compaction
        'delta.logRetentionDuration'    = 'interval 30 days',
        'delta.deletedFileRetentionDuration' = 'interval 7 days'
    )
""")

## Key Takeaways

| Operation | SQL | DeltaTable API |
|---|---|---|
| Create table | `CREATE TABLE ... USING DELTA` | `.write.format("delta").saveAsTable()` |
| Insert | `INSERT INTO` | `.write.mode("append")` |
| Update | `UPDATE ... SET ... WHERE` | `delta_tbl.update(condition, set)` |
| Delete | `DELETE FROM ... WHERE` | `delta_tbl.delete(condition)` |
| Upsert | `MERGE INTO ... WHEN MATCHED ... WHEN NOT MATCHED` | `delta_tbl.merge(...).whenMatched...execute()` |
| History | `DESCRIBE HISTORY table` | `DeltaTable.history()` |
| Time travel | `SELECT ... VERSION AS OF n` | `.option("versionAsOf", n)` |
| Restore | `RESTORE TABLE ... TO VERSION AS OF n` | — |
| Compaction | `OPTIMIZE table ZORDER BY (col)` | — |
| Cleanup | `VACUUM table` | — |
| CDC | `readChangeFeed = true` | — |

**Unity Catalog naming:** `catalog.schema.table` — always use the three-part name in production.